# Der letzte Garten
### Ein Python-Datenrätsel in fünf Etappen

**Dein Ziel:** Rette die letzte lebende Saatgutkapsel einer verlassenen Raumstation und bring sie nach **Nival**. Dort soll seit Jahren zum ersten Mal wieder etwas wachsen.

Du erreichst die Station mit einem kleinen Bergungsschiff. In den Fenstern brennt kein Licht. Nur ein Frachtterminal funktioniert noch. Darauf klebt ein Zettel:

> «Nicht alles, was sendet, ist echt. Nicht alles, was grün leuchtet, ist sicher. Bitte bring den Garten nach Hause.»

Unter Frostglas liegen sechs Kapseln. Du kannst **genau eine** retten. Dafür brauchst du ein echtes Zielsignal, die richtige Fracht, ein verlässliches Energiemodul, den Startcode und einen sicheren Kurs.

**Umfang:** etwa **75–110 Minuten** inklusive Hinweisen; fünf kleine Aufgaben, kein Zeitdruck und keine echte Countdown-Uhr. **Voraussetzung:** Python-Grundlagen und erste Pandas-Erfahrung. Kein Vorwissen über Raumfahrt nötig.

## So spielst du

Führe die **Startzelle einmal** aus. Bearbeite danach die fünf Etappen von oben nach unten: Daten ansehen, die `None`-Platzhalter mit deinem Code ersetzen, Prüfzelle ausführen. Die Aufgaben bauen aufeinander auf; erfolgreiche Prüfungen schalten jeweils ein Stück der Geschichte frei.

**Hinweise sind eingeklappt.** Nutze sie ruhig, sobald du einige Minuten festhängst. Ganz unten lassen sich einzelne Musterlösungen anzeigen, ohne sie automatisch auszuführen. Das Ziel ist der Aha-Moment, nicht das Raten der passenden Pandas-Funktion.

Nur `pandas` und dein vorhandenes Jupyter werden gebraucht. Alle Daten sind erfunden und im Notebook enthalten; es gibt keine Downloads, Dateizugriffe oder zusätzlichen Installationen. Arbeite auf Kopien der Rohdaten. **«Run All» ist ungefährlich**, löst aber nichts: Unbearbeitete Aufgaben melden nur «Noch offen». Ein erneuter Lauf der Startzelle setzt die Statusanzeigen zurück.

In [ ]:
import html
import pandas as pd
from IPython.display import HTML, Markdown, display

# Alle Daten sind erfunden. Keine Downloads, kein Netzwerk, keine Dateien nötig.
funk = pd.DataFrame([
    ("F01", " nav-a ",    "aktiv",    "1'280 mW"),
    ("F02", "NAV-B",      " AKTIV ",  "2 570 mW"),
    ("F03", "Nav-C",      "aktiv",    "2’740,0 mW"),
    ("F04", "NAV-D ",     "aktiv",    "2\u202f660 mW"),
    ("F05", "NAV-C-TEST", "aktiv",    "9'999 mW"),
    ("F06", "SIM-NAV-A",  "aktiv",    "8'888 mW"),
    ("F07", "NAV-A",      "inaktiv",  "9'100 mW"),
    ("F08", "NAV-B",      "aktiv",    "--"),
    ("F09", "NAV-D",      "aktiv",    "0 mW"),
    ("F10", "NAV-AA",     "aktiv",    "9'800 mW"),
    ("F11", "NAV-C",      "aktiv",    "-12 mW"),
    ("F12", "NAV-B",      "wartung",  "8'500 mW"),
], columns=["paket_id", "sender", "status", "pegel_roh"])

manifest = pd.DataFrame([
    ("K-01", "Saatgut", "NAV-A", "SA-14"),
    ("K-02", "Saatgut", "NAV-C", "SC-81"),
    ("K-03", "Saatgut", "NAV-C", "SC-72"),
    ("K-04", "Arznei",  "NAV-C", "MD-30"),
    ("K-05", "Saatgut", "NAV-C", "SC-60"),
    ("K-06", "Saatgut", "NAV-D", "SD-28"),
], columns=["kapsel_id", "ladung", "zielsignal", "plombe_soll"])
scans = pd.DataFrame([
    (" k-01", "SA-14"),
    ("K-02 ", "SC-18"),
    ("k-03",  "SC-72"),
    ("k-04 ", "MD-30"),
    ("K-06",  "SD-28"),
    ("k-03",  "SC-72"),  # identische Funkwiederholung
    ("K-99",  "XX-99"),
], columns=["kapsel_id", "plombe_ist"])

# Einzelversuche, keine voraggregierten Prozentsätze.
_testzeilen = []
for _modul, _last, _anzahl, _erfolge in [
    ("Aster", "niedrig", 3, 3), ("Aster", "hoch", 10, 7),
    ("Boreal", "niedrig", 10, 8), ("Boreal", "hoch", 5, 4),
    ("Cirrus", "niedrig", 1, 1), ("Cirrus", "hoch", 1, 1),
]:
    for _i in range(_anzahl):
        _nr = len(_testzeilen) + 1
        _testzeilen.append((
            f"T{_nr:02}", _modul, _last, int(_i < _erfolge),
            "protokolliert" if _nr % 3 == 0 else None,
        ))
tests = pd.DataFrame(
    _testzeilen, columns=["test_id", "modul", "last", "erfolg", "notiz"]
).sample(frac=1, random_state=17).reset_index(drop=True)

pulse = pd.DataFrame([
    (0, "oben", 2), (0, "unten", 4),
    (1, "oben", 1), (1, "oben", 2), (1, "unten", 2),
    (2, "oben", 4),
    # Takt 3 bleibt völlig still.
    (4, "oben", 1), (4, "unten", 1),
    (5, "oben", 3), (5, "unten", 4),
    (6, "oben", 2),
    (7, "oben", 4), (7, "unten", 1),
], columns=["takt", "kanal", "impulse"]).sample(
    frac=1, random_state=9
).reset_index(drop=True)

routen = pd.DataFrame([
    ("Dämmerung",  "NAV-C", 29, 23),
    ("Nordlicht",  "NAV-C", 24, 22),
    ("Morgenwind", "NAV-C", 35, 16),
    ("Sternbogen", "NAV-A", 18, 12),
    ("Echo",       "NAV-C", 28, 28),
], columns=["kurs", "zielsignal", "dauer_min", "bedarf"]).set_index("kurs")
rueckgewinnung = pd.Series(
    [8, 0, 3, 2, 5],
    index=pd.Index(["Echo", "Sternbogen", "Morgenwind", "Nordlicht", "Dämmerung"], name="kurs"),
    name="rueckgewinnung",
)
kapselgrenzen = pd.DataFrame(
    {"kuehlfenster_min": [27, 24, 31, 45, 20, 38]},
    index=pd.Index([f"K-{i:02}" for i in range(1, 7)], name="kapsel_id"),
)
modulleistung = pd.DataFrame(
    {"startenergie": [27, 19, 34]},
    index=pd.Index(["Aster", "Boreal", "Cirrus"], name="modul"),
)

# Prüflogik. Ergebnisse werden unabhängig von der Musterlösung verglichen.
_status = {n: False for n in range(1, 6)}
_titel = {1: "Signal", 2: "Kapsel", 3: "Energie", 4: "Taktcode", 5: "Kurs"}
_belohnung = {
    1: "Das echte Leuchtfeuer antwortet. Nival ist noch da. Die Frachtschleuse gibt den nächsten Datensatz frei.",
    2: "Die richtige Kapsel löst sich aus ihrer Halterung. Hinter dem Sichtfenster: winzige grüne Keimlinge.",
    3: "Das gewählte Modul fährt hoch. Zum ersten Mal seit Jahren leuchtet der Hangar. Die Kapsel bleibt gekühlt.",
    4: "Die acht Takte rasten ein. Die Startverriegelung öffnet sich; am Terminal erscheint die Kurskarte.",
    5: "Eine Route bleibt übrig. Die Energie reicht, die Kühlung hält, das Ziel stimmt. Du kannst starten.",
}

def _karte(titel, text, erfolg=False):
    rand = "#2f7a65" if erfolg else "#788697"
    display(HTML(
        f'<div style="border-left:4px solid {rand};padding:10px 14px;margin:8px 0;line-height:1.5">'
        f'<strong>{html.escape(titel)}</strong><br>{html.escape(text)}</div>'
    ))

def _gleich(ist, soll):
    try:
        pd.testing.assert_frame_equal(
            ist, soll, check_dtype=False, check_names=False,
            check_categorical=False, check_exact=False, rtol=1e-8, atol=1e-8,
        )
        return True
    except (AssertionError, TypeError, ValueError):
        return False

def _fehlende_spalten(df, spalten):
    return not isinstance(df, pd.DataFrame) or not set(spalten).issubset(df.columns)

def _validiere(n, *a):
    if any(x is None for x in a):
        return "Noch offen: Ersetze die None-Platzhalter in deiner Lösungszelle."
    try:
        if n == 1:
            df, signal = a
            sp = ["paket_id", "sender", "pegel"]
            if _fehlende_spalten(df, sp):
                return "Erwartet wird ein DataFrame mit paket_id, sender und pegel."
            if not pd.api.types.is_numeric_dtype(df["pegel"]):
                return "pegel muss numerisch sein, nicht Text. Ungültige Messwerte sind keine Nullmessungen."
            soll = pd.DataFrame([
                ("F01", "NAV-A", 1280), ("F02", "NAV-B", 2570),
                ("F03", "NAV-C", 2740), ("F04", "NAV-D", 2660),
            ], columns=sp)
            ist = df[sp].sort_values("paket_id").reset_index(drop=True)
            if not _gleich(ist, soll):
                return "Die bereinigten Signale stimmen noch nicht. Prüfe den vollständigen Sendernamen, Status und positive Messwerte."
            if not isinstance(signal, str) or signal != "NAV-C":
                return "Die Tabelle stimmt. Wähle daraus den Sender mit dem höchsten numerischen Pegel."
        elif n == 2:
            df, kapsel = a
            sp = ["kapsel_id", "ladung", "zielsignal", "plombe_soll", "plombe_ist", "_merge"]
            if _fehlende_spalten(df, sp):
                return "frachtpruefung benötigt die Spalten beider Tabellen und _merge."
            if df["kapsel_id"].duplicated().any():
                return "Eine Kapsel steht mehrfach im Ergebnis. Entferne echte Scan-Wiederholungen und prüfe die 1:1-Verknüpfung."
            soll = pd.DataFrame([
                ("K-01", "Saatgut", "NAV-A", "SA-14", "SA-14", "both"),
                ("K-02", "Saatgut", "NAV-C", "SC-81", "SC-18", "both"),
                ("K-03", "Saatgut", "NAV-C", "SC-72", "SC-72", "both"),
                ("K-04", "Arznei",  "NAV-C", "MD-30", "MD-30", "both"),
                ("K-05", "Saatgut", "NAV-C", "SC-60", None,    "left_only"),
                ("K-06", "Saatgut", "NAV-D", "SD-28", "SD-28", "both"),
                ("K-99", None,      None,    None,    "XX-99", "right_only"),
            ], columns=sp).astype("string")
            ist = df[sp].sort_values("kapsel_id").reset_index(drop=True).astype("string")
            if not _gleich(ist, soll):
                return "Die Frachtprüfung stimmt noch nicht. Keine Seite darf verschwinden; auch fehlende Gegenstücke müssen sichtbar bleiben."
            if not isinstance(kapsel, str) or kapsel != "K-03":
                return "Die Verknüpfung stimmt. Prüfe gleichzeitig: Saatgut, richtiges Ziel, beide Einträge und passende Plombe."
        elif n == 3:
            gruppen, gesamt, modulname = a
            sp = ["erfolge", "versuche", "quote"]
            if _fehlende_spalten(gruppen, sp) or _fehlende_spalten(gesamt, sp):
                return "Beide Ergebnistabellen brauchen erfolge, versuche und quote."
            if not isinstance(gruppen.index, pd.MultiIndex) or gruppen.index.names != ["modul", "last"]:
                return "statistik braucht einen MultiIndex in der Reihenfolge modul, last."
            index = pd.MultiIndex.from_tuples([
                ("Aster", "hoch"), ("Aster", "niedrig"),
                ("Boreal", "hoch"), ("Boreal", "niedrig"),
                ("Cirrus", "hoch"), ("Cirrus", "niedrig"),
            ], names=["modul", "last"])
            soll_gr = pd.DataFrame([
                (7, 10, .7), (3, 3, 1), (4, 5, .8),
                (8, 10, .8), (1, 1, 1), (1, 1, 1),
            ], index=index, columns=sp)
            if not _gleich(gruppen[sp].sort_index(), soll_gr):
                return "Die Laststufen stimmen noch nicht. Jeder Test zählt, auch ohne Notiz. Eine Quote liegt zwischen 0 und 1."
            soll_ge = pd.DataFrame([
                (10, 13, 10/13), (12, 15, 12/15), (2, 2, 1),
            ], index=pd.Index(["Aster", "Boreal", "Cirrus"], name="modul"), columns=sp)
            if not _gleich(gesamt[sp].sort_index(), soll_ge):
                return "Prüfe die Gesamtquote: Alle Einzeltests haben dasselbe Gewicht, nicht die beiden Laststufen. Noch nicht runden."
            if not isinstance(modulname, str) or modulname != "Boreal":
                return "Die Tabellen stimmen. Wähle die höchste Gesamtquote unter den ausreichend getesteten Modulen."
        elif n == 4:
            df, code = a
            if _fehlende_spalten(df, ["oben", "unten"]):
                return "impulsraster braucht die Spalten oben und unten."
            if list(df.index) != list(range(8)) or list(df.columns) != ["oben", "unten"]:
                return "Das Raster muss die Takte 0 bis 7 in dieser Reihenfolge und genau oben, unten enthalten."
            soll = pd.DataFrame(
                [[2,4], [3,2], [4,0], [0,0], [1,1], [3,4], [2,0], [4,1]],
                index=pd.Index(range(8), name="takt"), columns=["oben", "unten"],
            )
            if not _gleich(df, soll):
                return "Das Raster stimmt noch nicht. Pulse pro Takt und Kanal addieren; wirklich stille Takte und Kanäle mit 0 ergänzen."
            if not isinstance(code, str) or len(code) != 8:
                return "Der Taktcode ist ein Text aus genau acht Ziffern. Auch eine führende Null gehört dazu."
            if code != "07403126":
                return "Das Raster stimmt. Prüfe die Formel, die zeitliche Reihenfolge und die Umwandlung der einzelnen Ziffern."
        elif n == 5:
            df, kursname = a
            sp = ["zielsignal", "dauer_min", "bedarf", "nettoenergie"]
            if _fehlende_spalten(df, sp):
                return "kurspruefung muss alle ursprünglichen Routenspalten und nettoenergie enthalten."
            soll = pd.DataFrame([
                ("Dämmerung", "NAV-C",29,23,18), ("Nordlicht", "NAV-C",24,22,20),
                ("Morgenwind","NAV-C",35,16,13), ("Sternbogen","NAV-A",18,12,12),
                ("Echo",      "NAV-C",28,28,20),
            ], columns=["kurs"] + sp).set_index("kurs").sort_index()
            if not _gleich(df[sp].sort_index(), soll):
                return "Die Energiebilanz stimmt noch nicht. Rückgewinnung gehört zum Kursnamen, nicht zur zufälligen Zeilenposition."
            if not isinstance(kursname, str) or kursname != "Dämmerung":
                return "Die Bilanz stimmt. Prüfe Zielsignal, Kühlfenster deiner Kapsel und Energie deines Moduls gleichzeitig."
    except (TypeError, ValueError, KeyError, AttributeError, IndexError):
        return "Die Ausgabe lässt sich noch nicht prüfen. Kontrolliere Spalten, Index und die verlangten Datentypen."
    return None

def _pruefen(n, *werte):
    fehler = _validiere(n, *werte)
    _status[n] = fehler is None
    if fehler:
        _karte(f"{n}/5 · {_titel[n]}", fehler)
    else:
        _karte(f"{n}/5 · {_titel[n]} freigeschaltet", _belohnung[n], erfolg=True)

# Öffentliche Prüffunktionen: Auch None-Platzhalter sind erlaubt.
def pruefe_1(saubere_signale, leuchtfeuer):
    _pruefen(1, saubere_signale, leuchtfeuer)

def pruefe_2(frachtpruefung, kapsel_id):
    _pruefen(2, frachtpruefung, kapsel_id)

def pruefe_3(statistik, modulvergleich, modul):
    _pruefen(3, statistik, modulvergleich, modul)

def pruefe_4(impulsraster, taktcode):
    _pruefen(4, impulsraster, taktcode)

def pruefe_5(kurspruefung, kurs):
    _pruefen(5, kurspruefung, kurs)

def missionsstatus():
    text = " | ".join(
        f"{'✓' if _status[n] else '○'} {_titel[n]}" for n in range(1, 6)
    )
    _karte(f"Missionsstand: {sum(_status.values())}/5", text, erfolg=all(_status.values()))

def starte_rettung(gartenname="Morgenrot"):
    # Aktuelle Ergebnisse erneut prüfen: Alte grüne Haken reichen nicht.
    namen = {
        1: ("saubere_signale", "leuchtfeuer"),
        2: ("frachtpruefung", "kapsel_id"),
        3: ("statistik", "modulvergleich", "modul"),
        4: ("impulsraster", "taktcode"),
        5: ("kurspruefung", "kurs"),
    }
    for n, variablen in namen.items():
        fehler = _validiere(n, *(globals().get(v) for v in variablen))
        _status[n] = fehler is None
    if not all(_status.values()):
        offen = ", ".join(str(n) for n, ok in _status.items() if not ok)
        _karte("Die Startverriegelung wartet", f"Prüfe noch Aufgabe {offen}. Deine bisherigen Ergebnisse bleiben erhalten.")
        return
    if not isinstance(gartenname, str) or not gartenname.strip():
        _karte("Ein Name fehlt noch", "Gib deinem geretteten Garten einen Namen als Text.")
        return
    name = html.escape(gartenname.strip())
    display(HTML(f'''<div style="border:2px solid #2f7a65;border-radius:10px;padding:22px;line-height:1.65">
    <div style="font-size:12px;letter-spacing:2px">NIVAL · RETTUNG BESTÄTIGT · 5 / 5</div>
    <h2 style="margin:8px 0">Garten {name}</h2>
    <p>Das Schiff schiebt sich aus dem Hangar. Unter dir zieht die dunkle Station vorbei.
    Die Landebaken von Nival erscheinen genau dort, wo du sie berechnet hast.</p>
    <p><strong>29 Minuten später.</strong> Mit einer Energieeinheit Reserve und zwei Minuten
    Restkühlung setzt du auf. Eine Technikerin hebt die Kapsel aus der Halterung.
    »Die sind wirklich noch am Leben«, sagt sie.</p>
    <p>Drei Monate später hängt dein Gartenname über der Tür des neuen Gewächshauses.
    Ein Kind drückt dir die erste reife Erdbeere in die Hand. Auf dem Beipackzettel steht:</p>
    <blockquote>»Für die Person, die uns den Frühling zurückgebracht hat.«</blockquote>
    <p><strong>MISSION ERFÜLLT.</strong> Lebendes Saatgut gerettet. Gewächshaus neu gestartet.
    Du hast dir einen Platz an der ersten Ernte verdient.</p>
    </div>'''))

# Musterlösungen werden nur auf ausdrücklichen Aufruf angezeigt, niemals ausgeführt.
_LOESUNGEN = {1: '# 1. Neue Tabelle anlegen; Rohdaten bleiben unverändert.\nsignale = funk.copy()\nsignale["sender"] = signale["sender"].str.strip().str.upper()\nsignale["status"] = signale["status"].str.strip().str.lower()\n\n# Einheit, Leerzeichen und beide Apostroph-Arten entfernen.\npegeltext = (\n    signale["pegel_roh"].str.replace("mW", "", regex=False)\n    .str.replace(r"[\\s\'’]", "", regex=True)\n    .str.replace(",", ".", regex=False)\n)\nsignale["pegel"] = pd.to_numeric(pegeltext, errors="coerce")\n\nist_echt = signale["sender"].str.fullmatch(r"NAV-[A-D]", na=False)\nist_aktiv = signale["status"].eq("aktiv")\nist_messbar = signale["pegel"].gt(0)\nsaubere_signale = signale.loc[\n    ist_echt & ist_aktiv & ist_messbar,\n    ["paket_id", "sender", "pegel"],\n].copy()\nleuchtfeuer = saubere_signale.loc[\n    saubere_signale["pegel"].idxmax(), "sender"\n]\n', 2: 'fracht = manifest.copy()\nscanner = scans.copy()\nfor tabelle in (fracht, scanner):\n    tabelle["kapsel_id"] = tabelle["kapsel_id"].str.strip().str.upper()\n\n# Nur identische Wiederholungen entfernen, keine Schlüsselkollisionen verstecken.\nscanner = scanner.drop_duplicates()\nfrachtpruefung = fracht.merge(\n    scanner,\n    on="kapsel_id",\n    how="outer",\n    validate="one_to_one",\n    indicator=True,\n)\ngeeignet = (\n    frachtpruefung["_merge"].eq("both")\n    & frachtpruefung["ladung"].eq("Saatgut")\n    & frachtpruefung["zielsignal"].eq(leuchtfeuer)\n    & frachtpruefung["plombe_soll"].eq(frachtpruefung["plombe_ist"])\n)\ntreffer = frachtpruefung.loc[geeignet, "kapsel_id"]\nassert len(treffer) == 1, "Es muss genau eine passende Kapsel geben."\nkapsel_id = treffer.iloc[0]\n', 3: 'statistik = tests.groupby(["modul", "last"]).agg(\n    erfolge=("erfolg", "sum"),\n    versuche=("erfolg", "size"),\n)\nstatistik["quote"] = statistik["erfolge"] / statistik["versuche"]\n\n# Über die Laststufen zählen, nicht deren Prozentwerte mitteln.\nmodulvergleich = statistik.groupby(level="modul")[["erfolge", "versuche"]].sum()\nmodulvergleich["quote"] = (\n    modulvergleich["erfolge"] / modulvergleich["versuche"]\n)\nzugelassen = modulvergleich.loc[modulvergleich["versuche"].ge(10)]\nmodul = zugelassen["quote"].idxmax()\n', 4: 'impulsraster = (\n    pulse.pivot_table(\n        index="takt", columns="kanal", values="impulse",\n        aggfunc="sum", fill_value=0,\n    )\n    .reindex(index=range(8), columns=["oben", "unten"], fill_value=0)\n)\nziffern = (\n    (impulsraster["oben"] + 2 * impulsraster["unten"]) % 10\n).astype(int)\n# Zuerst einzelne Ziffern zu Text machen; sonst verschwindet eine führende Null.\ntaktcode = "".join(ziffern.astype(str))\n', 5: 'kurspruefung = routen.copy()\n# Pandas ordnet die Werte über die Kursnamen zu, nicht über die Zeilenposition.\nkurspruefung["nettoenergie"] = (\n    kurspruefung["bedarf"] - rueckgewinnung\n)\nbudget = modulleistung.loc[modul, "startenergie"]\nzeitlimit = kapselgrenzen.loc[kapsel_id, "kuehlfenster_min"]\n\nsicher = (\n    kurspruefung["zielsignal"].eq(leuchtfeuer)\n    & kurspruefung["dauer_min"].le(zeitlimit)\n    & kurspruefung["nettoenergie"].le(budget)\n)\npassende_kurse = kurspruefung.loc[sicher]\nassert len(passende_kurse) == 1, "Es muss genau einen sicheren Kurs geben."\nkurs = passende_kurse.index[0]\n'}

def zeige_loesung(aufgabe=0):
    if aufgabe == 0:
        print("Keine Lösung geöffnet. Für eine Musterlösung: zeige_loesung(1), …, zeige_loesung(5).")
        return
    if isinstance(aufgabe, bool) or not isinstance(aufgabe, int) or aufgabe not in _LOESUNGEN:
        print("Bitte eine Aufgabennummer von 1 bis 5 angeben; 0 zeigt keine Lösung.")
        return
    display(Markdown(f"### Musterlösung · Aufgabe {aufgabe}\n```python\n{_LOESUNGEN[aufgabe]}\n```"))

_karte("Verbindung zur Station hergestellt", "Daten und Prüffunktionen sind bereit. Beginne bei Aufgabe 1. Keine echte Uhr läuft.")

<details>
<summary><strong>Optional: 60-Sekunden-Werkzeugprobe</strong></summary>
<p>Bei <code>s = pd.Series([" 7 ", "?", "11"])</code> macht <code>pd.to_numeric(s.str.strip(), errors="coerce")</code> aus den gültigen Texten Zahlen und aus <code>?</code> einen fehlenden Wert. Mit <code>.notna()</code> findest du gültige Werte. Ein fehlender Wert ist nicht automatisch eine gemessene Null.</p>
<p><code>df.loc[maske, ["spalte"]]</code> wählt Zeilen und Spalten; <code>.copy()</code> hält die Rohdaten unverändert. Bedingungen kombinierst du mit <code>&amp;</code> und Klammern, nicht mit <code>and</code>.</p>
</details>

## 1 · Das echte Signal
*10–15 Minuten · Textbereinigung, Datentypen, Regex*

Das Funkgerät empfängt mehrere Leuchtfeuer. Einige sind Simulationen, andere abgeschaltet. Nivals echter Sender ist der **stärkste noch gültige Sender**.

Ein gültiges Paket erfüllt gleichzeitig: Nach Entfernen äusserer Leerzeichen und Vereinheitlichen der Schreibweise lautet der Sender **genau `NAV-` plus ein Buchstabe von A bis D**, sein Status ist **`aktiv`** und sein Pegel ist eine **gültige Zahl grösser als 0**. Mehrere Buchstaben oder zusätzliche Präfixe/Suffixe sind ungültig.

Im Pegeltext sind Leerzeichen und Apostrophe Tausendertrenner, das Komma ist ein Dezimaltrenner, `mW` ist die Einheit. `--` ist keine Messung. Sendernamen sollen am Ende grossgeschrieben sein.

**Deine Abgabe:** `saubere_signale` mit den Spalten `paket_id`, `sender`, `pegel` und **allen** gültigen Paketen; `leuchtfeuer` als Sendername mit dem höchsten numerischen Pegel. Die Zeilenreihenfolge ist egal. Leite das Ergebnis aus den Daten ab, statt den Namen einzutippen.

In [ ]:
display(funk)

In [ ]:
# Dein Code: funk nicht verändern.
saubere_signale = None
leuchtfeuer = None

In [ ]:
pruefe_1(saubere_signale, leuchtfeuer)

<details>
<summary><strong>Hinweis 1 · Nur ein kleiner Schubs</strong></summary>
<p>Bereinige Text und Zahlen zuerst, wähle danach die gültigen Zeilen. «Enthält NAV» reicht nicht: Ein Testsender kann damit ebenfalls durchrutschen.</p>
</details>
<details>
<summary><strong>Hinweis 2 · Konkrete Werkzeuge</strong></summary>
<p>Nützlich sind <code>.str.strip()</code>, <code>.str.upper()</code>, <code>.str.fullmatch(r"NAV-[A-D]")</code>, <code>.str.replace()</code>, <code>pd.to_numeric(..., errors="coerce")</code> und <code>.idxmax()</code>. Das Muster <code>[\s'’]</code> entfernt Leerzeichen und beide Apostroph-Arten. Bei <code>str.replace</code> dafür <code>regex=True</code> setzen.</p>
</details>

## 2 · Der falsche Frachtbrief
*15–20 Minuten · Dubletten, Merge, fehlende Gegenstücke*

Die Kapseln sehen identisch aus. Das Manifest nennt den Inhalt und die erwartete Plombe; der Scanner zeigt, was tatsächlich am Dock liegt. Er hat ein Paket mehrfach gesendet. Ausserdem fehlt etwas – und etwas gehört gar nicht hierher.

Bereinige die Kapsel-IDs in **beiden Tabellen**: äussere Leerzeichen weg, Grossschreibung. Entferne danach nur **vollständig identische Scan-Zeilen**. Verbinde die Tabellen über `kapsel_id`, sodass **auch Datensätze ohne Gegenstück auf beiden Seiten erhalten bleiben**. Lasse Pandas dabei eine **1:1-Beziehung ausdrücklich prüfen**; nutze einen Verknüpfungsindikator namens `_merge`.

Gesucht ist die einzige Kapsel, die in **beiden Tabellen** vorkommt, **Saatgut** enthält, zu deinem `leuchtfeuer` gehört und deren tatsächliche Plombe mit der erwarteten übereinstimmt.

**Deine Abgabe:** `frachtpruefung` mit allen Spalten aus Manifest und Scan sowie `_merge`; `kapsel_id` als Text. Fehlende Gegenstücke müssen in der Prüfung sichtbar bleiben, auch wenn du sie für die Auswahl ausschliesst.

In [ ]:
display(manifest)
display(scans)

In [ ]:
# Dein Code: Nutze leuchtfeuer aus Aufgabe 1.
frachtpruefung = None
kapsel_id = None

In [ ]:
pruefe_2(frachtpruefung, kapsel_id)

<details>
<summary><strong>Hinweis 1 · Nur ein kleiner Schubs</strong></summary>
<p>Ein innerer Join verschluckt genau die verdächtigen Fälle ohne Gegenstück. Ein normaler Merge kann dagegen einen Datensatz vervielfachen, wenn ein Schlüssel mehrfach vorkommt.</p>
</details>
<details>
<summary><strong>Hinweis 2 · Konkrete Werkzeuge</strong></summary>
<p>Nach der Normalisierung hilft <code>drop_duplicates()</code> ohne Schlüsselauswahl. Verknüpfe dann mit <code>how="outer", validate="one_to_one", indicator=True</code>. Der Indikator unterscheidet <code>both</code>, <code>left_only</code> und <code>right_only</code>. Vergleiche die beiden Plombenspalten zeilenweise.</p>
</details>

## 3 · Die trügerischen Prozente
*20–25 Minuten · Groupby, MultiIndex, korrekt gewichten*

Drei Energiemodule stehen bereit. Ein altes Etikett behauptet: «Cirrus – 100 % zuverlässig!» Du traust lieber den Einzeltests als dem Werbespruch.

Jede Zeile in `tests` ist **ein Versuch**, `erfolg` ist 1 oder 0. Eine fehlende `notiz` macht den Versuch nicht ungültig. Es gibt zwei Laststufen, die unterschiedlich oft getestet wurden.

Erstelle zuerst `statistik`: pro **Modul und Laststufe** die erfolgreichen Versuche, die Anzahl aller Versuche und die Erfolgsquote. Der Index soll ein **MultiIndex `modul`, `last`** sein. Erstelle daraus `modulvergleich`: dieselben Kennzahlen pro Modul über beide Laststufen hinweg, mit `modul` als Index. **Jeder Einzelversuch zählt gleich viel.**

Wähle das Modul mit der **höchsten Gesamtquote**, aber nur unter Modulen mit **mindestens zehn Versuchen insgesamt**.

**Deine Abgabe:** `statistik` und `modulvergleich`, jeweils mit den Spalten `erfolge`, `versuche`, `quote`; zusätzlich `modul` als Name. Quoten als Zahlen zwischen 0 und 1, **nicht vorzeitig runden**.

In [ ]:
# Vorschau; tests enthält sämtliche Versuche.
display(tests.head(10))
print(f"Insgesamt {len(tests)} Einzelversuche. Die volle Tabelle heisst tests.")

In [ ]:
# Dein Code: Gruppiere die vollständige Tabelle tests, nicht nur head(10).
statistik = None
modulvergleich = None
modul = None

In [ ]:
pruefe_3(statistik, modulvergleich, modul)

<details>
<summary><strong>Hinweis 1 · Nur ein kleiner Schubs</strong></summary>
<p>Eine Quote aus zwei Versuchen erzählt weniger als eine aus vielen. Und 100 % in einer kleinen Lastgruppe können 70 % in einer grossen Gruppe nicht einfach ausgleichen. Zähle zuerst.</p>
</details>
<details>
<summary><strong>Hinweis 2 · Konkrete Werkzeuge</strong></summary>
<p>Nutze benannte Aggregationen wie <code>versuche=("erfolg", "size")</code> und <code>erfolge=("erfolg", "sum")</code>. Über <code>groupby(level="modul")</code> kannst du die Zähler des MultiIndex zusammenfassen. Die Gesamtquote ist <code>Summe Erfolge / Summe Versuche</code>, nicht der Mittelwert der Gruppenquoten. <code>notiz.count()</code> würde Tests ohne Notiz übersehen.</p>
</details>

## 4 · Die dunkle Sekunde
*15–20 Minuten · Pivot, Reindex, geordnete Codes*

Der Starter sendet eine kurze Pulsfolge auf zwei Kanälen: **oben** und **unten**. Die Aufzeichnung ist durcheinander. Ein völlig stiller Takt fehlt sogar ganz. Der Mechaniker hat die Regel in den Staub geschrieben:

> «Acht Takte, von 0 bis 7. Addiere alle Pulse je Kanal und Takt. Die Ziffer ist `(oben + 2 × unten) modulo 10`. Lies von früh nach spät. Auch Stille zählt.»

Das Protokoll speichert **nur tatsächlich aufgetretene Pulse**. Kein Eintrag bedeutet **hier ausdrücklich 0**, nicht eine unbekannte Messung. Mehrere Einträge zum selben Kanal und Takt werden **addiert**, nicht gemittelt.

**Deine Abgabe:** `impulsraster` mit dem Index **0 bis 7 in dieser Reihenfolge** und genau den Spalten `oben`, `unten`, ohne fehlende Werte. Daraus `taktcode`: ein **Text mit acht Ziffern**, ohne Leerzeichen. Eine führende Null muss erhalten bleiben. In Python schreibt man den Modulo-Operator als `%`.

In [ ]:
display(pulse)

In [ ]:
# Dein Code: Stelle auch die vollständig fehlenden Takte wieder her.
impulsraster = None
taktcode = None

In [ ]:
pruefe_4(impulsraster, taktcode)

<details>
<summary><strong>Hinweis 1 · Nur ein kleiner Schubs</strong></summary>
<p>Du brauchst zuerst ein vollständiges Raster, erst danach die Ziffern. Ein Pivot kann fehlende Kombinationen auffüllen, erzeugt aber keinen Takt, der in den Eingangsdaten nirgends vorkommt.</p>
</details>
<details>
<summary><strong>Hinweis 2 · Konkrete Werkzeuge</strong></summary>
<p>Verwende <code>pivot_table(..., aggfunc="sum", fill_value=0)</code> und anschliessend <code>reindex(index=range(8), columns=["oben", "unten"], fill_value=0)</code>. Berechne dann eine Ziffer pro Zeile. <code>"".join(ziffern.astype(int).astype(str))</code> erhält die führende Null.</p>
</details>

## 5 · Der einzige Weg nach Hause
*10–15 Minuten · Index-Ausrichtung, Lookups, mehrere Bedingungen*

Das Schiff ist bereit. Fünf Kurse führen durch das Trümmerfeld. Der schnellste ist nicht zwingend sicher, der sparsamste nicht zwingend schnell genug.

`routen` nennt Zielsignal, Flugzeit und Energiebedarf. Die zusätzliche Messreihe `rueckgewinnung` ist **anders sortiert**: Ihre Werte gehören immer zum **gleichnamigen Kurs**, niemals einfach zur gleichen Zeilennummer.

Berechne `nettoenergie = bedarf − rueckgewinnung` für jeden Kurs. Die verfügbare Energie findest du in `modulleistung` für dein gewähltes `modul`, das Kühlfenster in `kapselgrenzen` für deine `kapsel_id`.

Ein sicherer Kurs hat gleichzeitig das richtige **`leuchtfeuer`**, eine Flugzeit **höchstens so lang wie das Kühlfenster** und eine Nettoenergie **höchstens so hoch wie die verfügbare Energie**. Genau einer erfüllt alles.

**Deine Abgabe:** `kurspruefung` mit **allen fünf Kursen**, den ursprünglichen Routenspalten und `nettoenergie`; der Kursname bleibt der Index. Speichere den einzig sicheren Kurs als Text in `kurs`.

In [ ]:
display(routen)
display(rueckgewinnung.to_frame())
display(modulleistung)
display(kapselgrenzen)

In [ ]:
# Dein Code: Nutze leuchtfeuer, kapsel_id und modul aus den früheren Aufgaben.
kurspruefung = None
kurs = None

In [ ]:
pruefe_5(kurspruefung, kurs)

<details>
<summary><strong>Hinweis 1 · Nur ein kleiner Schubs</strong></summary>
<p>Die Tabelle und die Messreihe haben schon passende Schlüssel: ihre Indexwerte. Wenn du diese durch Listen oder nackte Arrays ersetzt, geht die Zuordnung verloren.</p>
</details>
<details>
<summary><strong>Hinweis 2 · Konkrete Werkzeuge</strong></summary>
<p>Die direkte Rechnung <code>routen["bedarf"] - rueckgewinnung</code> richtet nach Kursnamen aus. Mit <code>.loc[modul, "startenergie"]</code> und <code>.loc[kapsel_id, "kuehlfenster_min"]</code> liest du deine Grenzen ab. Kombiniere danach alle drei Bedingungen mit <code>&amp;</code>.</p>
</details>

## Finale · Bring den Garten nach Hause

Alle fünf Prüfungen grün? **Gib dem neuen Garten einen Namen** und starte die Rettung. Der Bordcomputer prüft deine aktuellen Ergebnisse noch einmal; die Schlussnachricht erscheint nur bei einer vollständigen Lösung.

In [ ]:
missionsstatus()

In [ ]:
gartenname = "Morgenrot"  # Dein Name für Nivals neuen Garten.
starte_rettung(gartenname)

---
## Werkstatt · Nur bei Bedarf öffnen

**Gezielt nachsehen:** Setze unten die `0` auf die Nummer der Aufgabe, deren Musterlösung du sehen möchtest. Es wird nur Code angezeigt, **nichts ausgeführt oder überschrieben**. Übernimm und teste ihn in der betreffenden Lösungszelle. Spätere Aufgaben setzen deine früheren Ergebnisse voraus.

In [ ]:
zeige_loesung(0)

<details>
<summary><strong>Nach der Rettung · Was hinter den Rätseln steckte</strong></summary>
<p><strong>Signal:</strong> Erst normalisieren und in Zahlen umwandeln, dann filtern und vergleichen. Teiltreffer eines Regex sind nicht automatisch gültige Bezeichnungen.</p>
<p><strong>Fracht:</strong> Ein vollständiger Abgleich zeigt auch fehlende Gegenstücke. Identische Wiederholungen lassen sich entfernen; widersprüchliche Einträge desselben Schlüssels dürfen nicht einfach verschwinden. <code>validate="one_to_one"</code> ist dafür eine zusätzliche Sicherung.</p>
<p><strong>Module:</strong> <code>size</code> zählt Zeilen, <code>count</code> nicht fehlende Werte einer ausgewählten Spalte. Gesamtquoten entstehen aus passenden Zählern und Nennern, nicht blind aus dem Mittelwert von Teilquoten.</p>
<p><strong>Startcode:</strong> Aggregation und ein vollständiger Zielindex sind zwei getrennte Schritte. <code>reindex(..., fill_value=0)</code> füllt neu ergänzte Positionen, aber nicht pauschal schon vorhandene fehlende Werte. Fehlende Daten dürfen nur dann 0 werden, wenn ihre Bedeutung das rechtfertigt. Bei Zugangscodes ist Text oft der richtige Datentyp.</p>
<p><strong>Kurs:</strong> Pandas richtet Series nach Indexwerten aus. Diese Eigenschaft schützt vor falsch sortierten Zeilen – solange du die Schlüssel behältst.</p>
</details>

**Ein letzter Satz für dich:** Welcher Fehler hätte bei dir am ehesten zu einem falschen, aber plausibel aussehenden Ergebnis geführt?